# Explicit Randomized PCA - Implementation of the Rokhlin-Szlam-Tygert Algorithm

The existing notebooks (`RandomizedPCASmallDataSet` and `CIFAR10_PCA_vs_RandomizedPCA`)
treat randomized PCA as a black box through `PCA(svd_solver="randomized")`.
This notebook adds the missing piece from Section 2.2 of the proposal: an
**explicit NumPy implementation** of the randomized algorithm of Figure 1, so that
the individual roles of the random projection, the oversampling parameter and the
power iterations can be observed directly.

**What is added here**

1. A from-scratch `randomized_svd_explicit` following Figure 1 line by line.
2. `ExplicitRandomizedPCA`, a small scikit-learn-style wrapper (fit / transform / inverse_transform).
3. A deterministic NumPy baseline used to verify the intermediate matrix operations.
4. Validation on the Digits dataset against `PCA(svd_solver="full")` and `PCA(svd_solver="randomized")`.
5. Isolated studies of the oversampling parameter and the power-iteration count.
6. Run-to-run (seed) variability of the explicit implementation.

**A note on notation.** Figure 1 of the proposal uses `s` for oversampling and `p` for the
number of power iterations, while the text of Section 2.3 uses `p` for oversampling
(`l = k + p`) and `i` for the power-iteration count. To avoid confusion this notebook uses
explicit names throughout:

| This notebook | Figure 1 | Section 2.3 text | Meaning |
|---|---|---|---|
| `oversampling` | `s` | `p` | extra sampled directions, `l = k + oversampling` |
| `n_power_iter` | `p` | `i` | number of power iterations |

In [ ]:
import platform
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# Section 2.3 of the proposal asks for the runtime environment to be recorded so that
# the timing numbers can be interpreted later.
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Platform    :", platform.platform())
print("Processor   :", platform.processor() or "unknown")

## 1. The randomized algorithm

Given a centered matrix `A` of shape `(m, n)`, a target rank `k`, an oversampling
parameter `s` and a power-iteration count `p`:

```
1:  Omega = randn(n, k + s)
2:  Q     = orth(A Omega)
3:  for i = 1, 2, ..., p do
4:      G = orth(A^T Q)
5:      Q = orth(A G)
6:  end for
7:  B = Q^T A
8:  [U, S, V] = svd(B)
9:  U = Q U
10: U = U(:, 1:k), S = S(1:k, 1:k), V = V(:, 1:k)
```

The first two lines build an orthonormal basis `Q` for the range of a random projection of
`A`. Because `Omega` has `k + s` columns, `Q` spans slightly more directions than requested;
the extra `s` directions are the insurance that makes the dominant subspace very likely to
be captured. Lines 3-6 replace `A` by `(A A^T)^p A`, which raises every singular value to the
power `2p + 1` and therefore widens the gap between the wanted and unwanted parts of the
spectrum. This is the step that matters when the singular values decay slowly.

Line 7 forms the small `(k + s, n)` matrix `B`; the only SVD actually computed is the one on
line 8, and it is performed on this small matrix rather than on `A` itself. That is the entire
source of the speed-up.

`orth` is implemented with a thin QR factorisation, which is the standard and cheapest way to
obtain an orthonormal basis for the column space of a tall matrix.

In [ ]:
def orth(M):
    """Orthonormal basis for the column space of M, via a thin QR factorisation."""
    Q, _ = np.linalg.qr(M)
    return Q


def randomized_svd_explicit(A, k, oversampling=5, n_power_iter=0, seed=None,
                            return_basis=False):
    """Randomized SVD following Figure 1 of the proposal.

    Parameters
    ----------
    A : (m, n) array
        Input matrix. For PCA this must already be centered.
    k : int
        Target rank (number of components to return).
    oversampling : int
        Extra sampled directions; the sketch uses l = k + oversampling columns.
        This is s in Figure 1 and p in the Section 2.3 text.
    n_power_iter : int
        Number of power iterations. This is p in Figure 1 and i in the text.
    seed : int or None
        Seed for the Gaussian test matrix, so that runs are reproducible.
    return_basis : bool
        If True also return the sketch basis Q, for inspection.

    Returns
    -------
    U : (m, k), S : (k,), Vt : (k, n)
    """
    m, n = A.shape
    l = min(k + oversampling, min(m, n))       # width of the sketch
    rng = np.random.default_rng(seed)

    # 1: Omega = randn(n, k + s)
    Omega = rng.standard_normal((n, l))

    # 2: Q = orth(A Omega)
    Q = orth(A @ Omega)

    # 3-6: power iterations, re-orthonormalising at every half step for stability
    for _ in range(n_power_iter):
        G = orth(A.T @ Q)
        Q = orth(A @ G)

    # 7: B = Q^T A          (small: l x n)
    B = Q.T @ A

    # 8: [U, S, V] = svd(B)
    U_b, S, Vt = np.linalg.svd(B, full_matrices=False)

    # 9: U = Q U
    U = Q @ U_b

    # 10: truncate to the requested rank k
    U, S, Vt = U[:, :k], S[:k], Vt[:k, :]

    if return_basis:
        return U, S, Vt, Q
    return U, S, Vt

## 2. PCA wrappers

Both wrappers subtract the column means first, exactly as scikit-learn does, so that the
deterministic and randomized versions see the *same* centered matrix. Keeping the same
interface as scikit-learn (`fit_transform`, `inverse_transform`,
`explained_variance_ratio_`) means the new implementation can be dropped into the
comparisons used by the existing notebooks without changing anything else.

In [ ]:
class DeterministicPCA:
    """Baseline PCA from a full NumPy SVD of the centered matrix."""

    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        n_samples = X.shape[0]
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

        k = self.n_components
        self.singular_values_ = S[:k]
        self.components_ = Vt[:k]
        self.explained_variance_ = S[:k] ** 2 / (n_samples - 1)
        # total variance of the centered data = ||Xc||_F^2 / (n_samples - 1)
        total_var = (S ** 2).sum() / (n_samples - 1)
        self.explained_variance_ratio_ = self.explained_variance_ / total_var
        self._scores = U[:, :k] * S[:k]
        return self

    def fit_transform(self, X):
        return self.fit(X)._scores

    def transform(self, X):
        return (np.asarray(X, dtype=np.float64) - self.mean_) @ self.components_.T

    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_

    def reconstruct(self):
        """The rank-k approximation of the training data, U S V^T + mean."""
        return self.inverse_transform(self._scores)


class ExplicitRandomizedPCA:
    """PCA built on randomized_svd_explicit, with a scikit-learn-like interface."""

    def __init__(self, n_components, oversampling=10, n_power_iter=2, random_state=None):
        self.n_components = n_components
        self.oversampling = oversampling
        self.n_power_iter = n_power_iter
        self.random_state = random_state

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        n_samples = X.shape[0]
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        U, S, Vt = randomized_svd_explicit(
            Xc,
            k=self.n_components,
            oversampling=self.oversampling,
            n_power_iter=self.n_power_iter,
            seed=self.random_state,
        )

        self.singular_values_ = S
        self.components_ = Vt
        self.explained_variance_ = S ** 2 / (n_samples - 1)
        # The total variance is measured on the data, not on the approximation, so that
        # the ratio is directly comparable with the deterministic version.
        total_var = (Xc ** 2).sum() / (n_samples - 1)
        self.explained_variance_ratio_ = self.explained_variance_ / total_var
        self._scores = U * S
        return self

    def fit_transform(self, X):
        return self.fit(X)._scores

    def transform(self, X):
        return (np.asarray(X, dtype=np.float64) - self.mean_) @ self.components_.T

    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_

    def reconstruct(self):
        """The rank-k approximation of the training data, U S V^T + mean."""
        return self.inverse_transform(self._scores)

### A note on which reconstruction is measured

There are two different rank-k reconstructions one can form from a randomized fit, and they do
**not** agree:

* `U S V^T` - the approximation the algorithm actually produces. This is what `reconstruct()`
  returns and what the proposal means by "its rank-k reconstruction A_hat".
* `A V^T V` - the orthogonal projection of the data onto the computed directions. This is what
  you get by calling `transform` and then `inverse_transform`, and it is always at least as
  accurate, because the projection re-fits the coefficients optimally inside the subspace it was
  given.

For a deterministic SVD the two coincide exactly. For a randomized fit the second one flatters
the method, so mixing the two inside one comparison would produce numbers that cannot be placed
side by side. Every error in this notebook uses the first definition, which is also the one used
throughout `Synthetic_Scalability_MonteCarlo.ipynb`.

## 3. Validation on the Digits dataset

The Digits dataset (1797 samples, 64 features) is used exactly as in the existing small-dataset
notebook, with `k = 20`. The purpose here is correctness rather than speed: on a 1797x64 matrix
a full SVD is trivially cheap, so the explicit implementation is expected to be *slower*, and
only the agreement of the outputs is meaningful.

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target

k = 20
SEED = 42

print("Dataset shape:", X.shape)
print("Components   :", k)

methods = {}

# --- scikit-learn, deterministic --------------------------------------------------
start = time.perf_counter()
sk_full = PCA(n_components=k, svd_solver="full")
Z_sk_full = sk_full.fit_transform(X)
methods["sklearn full"] = (sk_full, Z_sk_full, time.perf_counter() - start)

# --- scikit-learn, randomized -----------------------------------------------------
start = time.perf_counter()
sk_rand = PCA(n_components=k, svd_solver="randomized", random_state=SEED)
Z_sk_rand = sk_rand.fit_transform(X)
methods["sklearn randomized"] = (sk_rand, Z_sk_rand, time.perf_counter() - start)

# --- our deterministic NumPy baseline ---------------------------------------------
start = time.perf_counter()
np_full = DeterministicPCA(n_components=k)
Z_np_full = np_full.fit_transform(X)
methods["numpy full (ours)"] = (np_full, Z_np_full, time.perf_counter() - start)

# --- our explicit randomized implementation ---------------------------------------
start = time.perf_counter()
ex_rand = ExplicitRandomizedPCA(n_components=k, oversampling=10,
                                n_power_iter=2, random_state=SEED)
Z_ex_rand = ex_rand.fit_transform(X)
methods["explicit randomized (ours)"] = (ex_rand, Z_ex_rand, time.perf_counter() - start)

Xc = X - X.mean(axis=0)

print()
header = "{:<28} {:>10} {:>11} {:>12} {:>14}".format(
    "method", "time (s)", "expl. var", "MSE", "rel. Fro err")
print(header)
print("-" * len(header))
for name, (model, Z, t) in methods.items():
    # Z is the score matrix U*S returned by fit_transform, for every method alike, so
    # inverse_transform(Z) is the rank-k approximation U S V^T + mean in all four cases.
    X_hat = model.inverse_transform(Z)
    mse = np.mean((X - X_hat) ** 2)
    rel = np.linalg.norm(X - X_hat, "fro") / np.linalg.norm(Xc, "fro")
    ev = model.explained_variance_ratio_.sum()
    print(f"{name:<28} {t:>10.6f} {ev:>11.6f} {mse:>12.6f} {rel:>14.6f}")

### 3.1 Do the four methods find the same subspace?

Comparing components entry by entry is misleading, because singular vectors are only defined up
to a sign (and, when singular values are nearly tied, up to a rotation inside the tied block).
Two more robust checks are used:

* **Singular values** - these are basis-independent, so they should agree to high precision.
* **Principal angles** between the two 20-dimensional component subspaces. If the largest
  principal angle is near zero the two methods span the same subspace, whatever basis they chose
  inside it.

In [ ]:
def largest_principal_angle(V1, V2):
    """Largest principal angle (radians) between the row spaces of V1 and V2."""
    Q1 = orth(V1.T)
    Q2 = orth(V2.T)
    s = np.linalg.svd(Q1.T @ Q2, compute_uv=False)
    return np.arccos(np.clip(s, -1.0, 1.0)).max()


ref = methods["sklearn full"][0]

header = "{:<28} {:>12} {:>17}".format("method", "max |dS|", "max angle (deg)")
print(header)
print("-" * len(header))
for name, (model, Z, t) in methods.items():
    ds = np.abs(model.singular_values_ - ref.singular_values_).max()
    ang = np.degrees(largest_principal_angle(model.components_, ref.components_))
    print(f"{name:<28} {ds:>12.3e} {ang:>17.6f}")

print()
print("Singular values (first 10)")
for name, (model, Z, t) in methods.items():
    print(f"  {name:<28}", np.array2string(model.singular_values_[:10],
                                           precision=3, suppress_small=True))

## 4. Effect of the oversampling parameter

`oversampling` controls how many directions beyond `k` are sampled. With `oversampling = 0` the
sketch has exactly `k` columns and there is no margin for the random projection to miss part of
the dominant subspace, so the approximation is noticeably worse. A handful of extra columns is
usually enough; beyond that the error curve flattens while the cost keeps growing.

Power iterations are switched off here (`n_power_iter = 0`) so that the effect of oversampling
is not masked.

In [ ]:
oversampling_grid = [0, 1, 2, 5, 10, 20, 40]
N_REPEAT = 5

# The truncated SVD is the best possible rank-k approximation, so its error is the
# yardstick every randomized run is measured against.
opt_err = np.linalg.norm(X - np_full.reconstruct(), "fro")

os_err, os_time = [], []
for s in oversampling_grid:
    model = ExplicitRandomizedPCA(n_components=k, oversampling=s,
                                  n_power_iter=0, random_state=SEED)
    model.fit(X)                       # warm-up run, excluded from the timing
    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        model.fit(X)
        times.append(time.perf_counter() - t0)
    err = np.linalg.norm(X - model.reconstruct(), "fro")
    os_err.append(err / opt_err)
    os_time.append(np.mean(times))
    print(f"oversampling = {s:>3}  l = {k + s:>3}   "
          f"err / optimal = {err / opt_err:.6f}   time = {np.mean(times) * 1000:7.3f} ms")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(oversampling_grid, os_err, "o-")
ax[0].axhline(1.0, color="k", ls="--", lw=1, label="optimal (truncated SVD)")
ax[0].set_xlabel("oversampling s")
ax[0].set_ylabel("Frobenius error / optimal error")
ax[0].set_title(f"Accuracy vs oversampling (Digits, k={k}, no power iterations)")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(oversampling_grid, np.array(os_time) * 1000, "o-", color="tab:red")
ax[1].set_xlabel("oversampling s")
ax[1].set_ylabel("mean fit time (ms)")
ax[1].set_title("Cost vs oversampling")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Effect of power iterations

Each power iteration multiplies by `A A^T`, so a singular value `sigma` is effectively replaced by
`sigma^(2p+1)`. Directions that are only slightly weaker than the wanted ones are suppressed much
faster than the wanted ones, which is why a small number of iterations recovers almost all of the
lost accuracy. The price is two extra matrix products and two extra QR factorisations per
iteration, so the cost grows roughly linearly in `n_power_iter`.

In [ ]:
power_grid = [0, 1, 2, 3]

pw_err, pw_time, pw_angle = [], [], []
for q in power_grid:
    model = ExplicitRandomizedPCA(n_components=k, oversampling=10,
                                  n_power_iter=q, random_state=SEED)
    model.fit(X)                       # warm-up
    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        model.fit(X)
        times.append(time.perf_counter() - t0)
    err = np.linalg.norm(X - model.reconstruct(), "fro")
    ang = np.degrees(largest_principal_angle(model.components_, ref.components_))
    pw_err.append(err / opt_err)
    pw_time.append(np.mean(times))
    pw_angle.append(ang)
    print(f"n_power_iter = {q}   err / optimal = {err / opt_err:.8f}   "
          f"max angle = {ang:8.4f} deg   time = {np.mean(times) * 1000:7.3f} ms")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(power_grid, pw_err, "o-")
ax[0].axhline(1.0, color="k", ls="--", lw=1)
ax[0].set_xlabel("number of power iterations")
ax[0].set_ylabel("Frobenius error / optimal error")
ax[0].set_title("Accuracy")
ax[0].set_xticks(power_grid)
ax[0].grid(True, alpha=0.3)

ax[1].semilogy(power_grid, np.maximum(pw_angle, 1e-12), "o-", color="tab:green")
ax[1].set_xlabel("number of power iterations")
ax[1].set_ylabel("largest principal angle (deg)")
ax[1].set_title("Subspace agreement with full SVD")
ax[1].set_xticks(power_grid)
ax[1].grid(True, alpha=0.3)

ax[2].plot(power_grid, np.array(pw_time) * 1000, "o-", color="tab:red")
ax[2].set_xlabel("number of power iterations")
ax[2].set_ylabel("mean fit time (ms)")
ax[2].set_title("Cost")
ax[2].set_xticks(power_grid)
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Monte Carlo behaviour on Digits

Randomized PCA is a Monte Carlo algorithm: it always performs the same sequence of operations,
but the Gaussian test matrix changes with the seed, so the accuracy is a random variable. The
cell below repeats the fit with many seeds and reports the spread of the reconstruction error,
with and without power iterations. The deterministic optimum is a single number and is shown as
a reference line.

In [ ]:
seeds = list(range(30))

results = {}
for q in [0, 2]:
    errs = []
    for s in seeds:
        model = ExplicitRandomizedPCA(n_components=k, oversampling=10,
                                      n_power_iter=q, random_state=s)
        model.fit(X)
        errs.append(np.linalg.norm(X - model.reconstruct(), "fro") / opt_err)
    errs = np.array(errs)
    results[q] = errs
    print(f"n_power_iter = {q}  over {len(seeds)} seeds:")
    print(f"    mean = {errs.mean():.8f}   std = {errs.std(ddof=1):.3e}   "
          f"min = {errs.min():.8f}   max = {errs.max():.8f}")

plt.figure(figsize=(9, 4))
for q, errs in results.items():
    plt.hist(errs, bins=12, alpha=0.6, label=f"n_power_iter = {q}")
plt.axvline(1.0, color="k", ls="--", lw=1, label="optimal (truncated SVD)")
plt.xlabel("Frobenius error / optimal error")
plt.ylabel("count")
plt.title(f"Seed-to-seed variation of the explicit randomized PCA (Digits, k={k})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Visual check

The same three-row layout as the existing small-dataset notebook, with the explicit
implementation in place of the scikit-learn randomized solver.

In [ ]:
X_det_rec = np_full.reconstruct()
X_exp_rec = ex_rand.reconstruct()

rows = [
    ("Original", [digits.images[i] for i in range(6)]),
    ("Deterministic (full SVD)", [X_det_rec[i].reshape(8, 8) for i in range(6)]),
    ("Explicit randomized", [X_exp_rec[i].reshape(8, 8) for i in range(6)]),
]

fig, axes = plt.subplots(3, 6, figsize=(12, 6.5))
for j, (title, imgs) in enumerate(rows):
    for i, img in enumerate(imgs):
        axes[j, i].imshow(img, cmap="gray")
        axes[j, i].axis("off")
    axes[j, 0].set_title(title, loc="left", fontsize=10)

plt.suptitle(f"Digits reconstructions, k = {k}")
plt.tight_layout()
plt.show()

## 8. First 16 principal directions as images

A second way to confirm that the explicit implementation recovers the same subspace: the leading
principal directions themselves should look the same (up to an overall sign) as those produced by
the full SVD.

In [ ]:
def sign_align(V, V_ref):
    """Flip the sign of each row of V so that it agrees with the reference."""
    signs = np.sign(np.sum(V * V_ref, axis=1))
    signs[signs == 0] = 1.0
    return V * signs[:, None]


V_det = np_full.components_
V_exp = sign_align(ex_rand.components_, V_det)

fig, axes = plt.subplots(4, 8, figsize=(13, 7))
for i in range(16):
    r, c = divmod(i, 4)
    axes[r, 2 * c].imshow(V_det[i].reshape(8, 8), cmap="RdBu")
    axes[r, 2 * c].set_title(f"det #{i + 1}", fontsize=8)
    axes[r, 2 * c + 1].imshow(V_exp[i].reshape(8, 8), cmap="RdBu")
    axes[r, 2 * c + 1].set_title(f"rand #{i + 1}", fontsize=8)
    axes[r, 2 * c].axis("off")
    axes[r, 2 * c + 1].axis("off")

plt.suptitle("Principal directions: deterministic (left of each pair) vs explicit randomized")
plt.tight_layout()
plt.show()

corr = np.abs(np.sum(V_exp * V_det, axis=1))
print("Per-component |cosine| with the deterministic directions:")
print(np.array2string(corr, precision=6, suppress_small=True))

## 9. Summary

* The explicit implementation of Figure 1 reproduces the scikit-learn and full-SVD results on
  Digits: the singular values agree to a small tolerance and the largest principal angle between
  the two 20-dimensional subspaces is close to zero once a couple of power iterations are used.
* **Oversampling** buys a large accuracy improvement for the first few extra columns and then
  saturates, while the cost keeps increasing - which is why a small fixed value such as 5 or 10
  is the usual choice.
* **Power iterations** reduce the error towards the deterministic optimum roughly geometrically,
  at a cost that grows about linearly. On Digits the singular values decay quickly, so even
  `n_power_iter = 1` is already close to optimal; the synthetic experiments in
  `Synthetic_Scalability_MonteCarlo.ipynb` show the harder slowly-decaying case where more
  iterations genuinely matter.
* The seed study confirms the Monte Carlo character of the method: the error is a random
  variable with a small but non-zero spread, and power iterations shrink both its mean and its
  spread.
* On a 1797x64 matrix the randomized method is *not* faster - the full SVD is already cheap and
  the sketch adds overhead. This is the expected small-matrix regime; the crossover is located in
  the companion scalability notebook.